# This notebook is an explenation of how to operate the MMEJ detection algorithm.
It contains examples to show both the Markov chain itself, and the MMEJ detection 
(including the Markov chain step).

<font size="5"> The notebook is divided into three parts: <br>
<font size="3"> 
Part A: Example of the algrithm preformance on a small sampled Soybean data. <br>
Part B: Fake examples of the MMEJ detection process. <br>
Part C: Proving the Markov chain integrety. <br>

<font size="5"> User must have:
<font size="3">
MMEJ_2nd_order_MM_v2 <br>
MicroHomology_module_v3 <br>
python 3.8.3 (or higher) <br>
numpy  1.19.5 <br>
pandas 1.3.1 <br>
biopython 1.79 (installation should include bio 0.5.0) <br>

In [1]:
# importing the relevant libraries/modules:
%load_ext autoreload
%autoreload 2

import datetime
import timeit
import pandas as pd
import numpy as np
import sys
sys.path.append('/home/labs/alevy/guyta/guy_master_project/src')
# from VcfProcess_module import VcfProcess
from MicroHomology_module_v3 import MicroHomology
from MMEJ_2nd_order_MM_v2 import motif_probabily_calc
pd.options.display.max_colwidth = 2400

# Part A: Looking at a small sampled Soybean dataset
This is also the actual pipline


In [2]:
# Loading the data:
# Specify Dtypes
dtypes = {'Unnamed: 0' : 'int64', 'fasta_context_position': 'str', 'REF' : 'str', 'ALT' : 'str', 
          'indel_type' : 'category', 'indel_length' : 'int64', 'indel_pos' : 'int64', 'Read' : 'str',
          'reference_context_seq_2K' : 'str', 'reference_context_seq_120' : 'str' }
# set data path
path_to_df = r'/home/labs/alevy/guyta/guy_master_project/data/soybean/Liu_et.al.2020/data_for_tests/20220208_sample_mmej_df.csv'
# loading data
df = pd.read_csv(path_to_df, sep = '\t', dtype = dtypes)
# droping non-relevant columns
df.drop(columns = ['Unnamed: 0', 'fasta_context_position'], inplace = True)
# renaming columns
df.rename(columns = {'fasta_context_seq_120': 'reference_context_seq_120',
                     'fasta_context_seq_2K': 'reference_context_seq_2K', 'ancesstral_allele' : 'REF', 'derived_allele' : 'ALT',
                     'accession_context' : 'Read'}, inplace = True)
# General information on the sample dataset
print(df.info())
df.sample(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype   
---  ------                     --------------  -----   
 0   REF                        160 non-null    object  
 1   ALT                        160 non-null    object  
 2   indel_type                 160 non-null    category
 3   indel_length               160 non-null    int64   
 4   indel_pos                  160 non-null    int64   
 5   Read                       160 non-null    object  
 6   reference_context_seq_2K   160 non-null    object  
 7   reference_context_seq_120  160 non-null    object  
dtypes: category(1), int64(2), object(5)
memory usage: 9.0+ KB
None


,REF,ALT,indel_type,indel_length,indel_pos,Read,reference_context_seq_2K,reference_context_seq_120
129,T,-,DEL,1,119,ACGCCCGAAATGGACTCGGATGAATGCACAAATTGATAAAAGAACATATTTTGGAAACATTGGGTTGATTAAAATAGAGGAAATGAATCCTGAGCCCTAGCATCACATGACCATAAAAATTGACACTTGAGTGTCCGCATAGGTGCATGCATGACCAGTTTTGCATAAAGTTTCCAAATCATCATTTTTGCATTTGTGTCATGGAAATAATGTGGGGCATCCCTTTTATCCTTGAACCA,GCCCCCACCACCTCATCGTGCTGGCCAGGCTGGGGCATTTGACATGGAGCAGTATTTACGGCATTTGGTTCGCCAGCAGGCGGCCAACCACCGAGCACATGTACGGACCCATGATTGTCTGTACCAGATGAGCCTTAGCATGCAGAGCCAGGGCTTCGCTCCTTCTTCATGCCCTACTCCAGACCAGTTCAGGGCAGAGGTTGCATGGCCCGGAGATTGGCCCGAGGCCCAAGCAGGAGAGGCACCCCCAGAAGCTCCCGGCGATGGAGAAGAAGCCCACGAGGATGAGGAGATGGCTGATTTGCTTGATTTCTTGGGAGGGAGTGGAGACACGTGACTGGGAGATCCCCAGATTCATGTTTTCTTTCATATTTCTTTTGTCATTTTTTTCTATGTTATTGTTTTGACTTGAGAGACTAATGTTTGTTTTTGTTGTTTCGATTGTCATTTGTACAGCGCATACATTTTTGTTTAGATTGGTGCGTTAGTATTTATATATCCTTACTATCGATGATGTTTGAAATTCTGGAACCGTGTAGAGTTCTTCGTTTAGGAACATCGTCCAAAAGTATATATGTAACATAAACAAAAAAATCATGATAAAAATAAAAATAGAAAAGGAAAGAAAATGAAATAGAAAAGAAAAGAAAGTGAAAAGAAAAAGAGGGCAAATAGGGGAAAAAGGGCAAGTAAGGAAGAGGGTAGAAAAAAAAAAAAAAAAAAAAAAAAAAGAAAAAAAATATAGGTTGTCTAGCTAAAAAACGACATGCTTTTGAAAAGAGACGATTCCCAACTTTTCTTTGAAAAAGTTCGTTGATCATAACCAATTTTTGGAAAATGTGTCTACACCTGAAGGGTGAATGCTGTGAAATTTCCCCGGACGCCCGAAATGGACTCGGATGAATGCACAAATTGATAAAAGAACATATTTTGGAAACATTGGGTTGATTAAAATAGAGGAAATGAATCCTGAGCCCTAGCATCACATGACCATAAAAATTTGACACTTGAGTGTCCGCATAGGTGCATGCATGACCAGTTTTGCATAAAGTTTCCAAATCATCATTTTTGCATTTGTGTCATGGAAATAATGTGGGGCATCCCTTTTATCCTTGAACCAAACCAAACCCTGACATGTATCATGTCTAGCCATTCTACAAGCTTTGAGCCAAAATCCTGACTCACCATAAACCTTGACCCAGGGTGAGAATGCCAATCCTTACCCTCGGAAGCAAAAAAAGAATAGAGGGGAAATTTCCAATCAAAGAAAAAGAGAAGGAAAATTTCCAATGAAAGCAAAAAAGAAATGAAGGAAAATTCCCCAATCAAAGAGTGGGAGAAAGCAAAAAAAAGGAAAAGAAGGAAAATTCCCCAATCAAAGAGTGGGAGAAAGCAAAAAGAAAAGAAAGGAAAATTCCCAATCAAAGAATGGGAGAAAGTAAAAAAGGAAGAAGAAGAAGGAAAGAAAGCTCCTGATCAAGGATCGAAAGAAACCAGAAGAAATGTGCAGAGAGGTCTTTGGACCAGACAATATCTGAACAGTACAGAATTGTCACCAAATGAACAAAAAGGAAGGAAAGGAAACCACGACCTAAAATGGTCTTCTCCCTTTAATTACCAACCAAAATCCCGTGCGCTAGCGACCTTTTTTTCCTCGCCCCGCACTAAACAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAGAAAAAAGAAAAAGCCAGAAAAATCAAAAGCCAAAAACACATAAAAACCGAAAGAAGAAACCATCAAAAGAACCCATTCCCAAGGGAAGCCCTATTGATCCATGATCACGCATGTAATTTTTGATTTGATAGGAAATAATTTGCAAAGTCAAGTCATGACATATCTATGGTTCGGAATTAGGATGAAACACTTACCTGTGCGAGATTGATACACTTTGAGTGGATTTCTTCTATTTTTGTCGAACCCAGTGTTTCCTCTAAATGGTCATTTAGAAACGAAATGCTAACATCCAA,ACGCCCGAAATGGACTCGGATGAATGCACAAATTGATAAAAGAACATATTTTGGAAACATTGGGTTGATTAAAATAGAGGAAATGAATCCTGAGCCCTAGCATCACATGACCATAAAAATTTGACACTTGAGTGTCCGCATAGGTGCATGCATGACCAGTTTTGCATAAAGTTTCCAAATCATCATTTTTGCATTTGTGTCATGGAAATAATGTGGGGCATCCCTTTTATCCTTGAACCA
73,-,TA,INS,2,121,TATCCTTAGTTTTTTATTTATAATGTGATTTAAAAGATAAGATGTTTGACTGATCTAGCATATTTTGGTATTAATAAAACCACGTGTATATATATATATATATATATGTATGTATATATATTAGTATATATATGTATGTATTTTTCTCGATGTGTGTCTGGGTGAAAGTGGTGATATGAATGGAATTGTCATTGTTATTTAAAAAATTATAGATTGGTTGTTGTTATTCTTGGTGGATTCTT,GGAAAGCTACCCTGAATTGCATGATTGCAGGTGGTCTCACAATTGGATATCTGATCTGCATGAACAAAAGATTTGCTAATGTCAAAAGTTTAAACCGGAAAAATAATGCATTGGAAGATATAAACTTTGTAAGAGTGGCTCCTTACTCTGGTTGAAGCGATGGAGGATGAGGTACTCATCTTGGGTTTTTCTGAAGTAAAAGTATGGTGATGTGATGTTAAGATGATGTTTGCAGAGGAACGGGTAATATGGGTTAATAGTTGAGTGCACCTGATTAATTGATTTATCTTTTCGATGGCATAAGTGGCGGTAGGTTTATGTTGGAAACGGTGTAAGTGAATAGTTGCTTCTTGAATGTGAAAGGCAGTGATGATATGAAGAGCCTCAACGTGAATAGTTGTGGAAGTAACCGAACGGTTTTTTTCTGAATGGTTTCAGAAGTGATTGGACTTTAGTGCTATTTGTGGTGTGCATATAATAGTTGGGCCTTAAATTCGTTAATTTTTTCCATGAGGTTTTCAGCTTAATAGATTAATTAATAGATTAATAATGTAAAGTTCACCTAATCATTAAAGTATTTTACACGTTCCCGTGCTTGAAGTAATTGCAGTATCAACTCTACCATTACCCATATCTTCTTAATTAATGCCCCTGTGAAACTGTTGTGTGTGCAGTTTCTAAAAAAATCAATAAAATGGAGGCACGTTAGAAAATAGAGAAGAATAGGCAAGATAGAGTAAAAAAACTGAAATTCAAACTGAAATAAATTAAAACAAAATTAAAAGGATTTTCTGTGAGGACTCTAATTCTCCCAGACAGATCAAGTTCCAAAGGATTGTAGTCCACTAACTACTACTACCGCTAGTTAGTCAGCCTTATTTATCCTTAGTTTTTTATTTATAATGTGATTTAAAAGATAAGATGTTTGACTGATCTAGCATATTTTGGTATTAATAAAACCACGTGTATATATATATATATATATATGTATGTATATATATGTATATATATGTATGTATTTTTCTCGATGTGTGTCTGGGTGAAAGTGGTGATATGAATGGAATTGTCATTGTTATTTAAAAAATTATAGATTGGTTGTTGTTATTCTTGGTGGATTCTTAGTATTTAAAATCAATGGAA

In [3]:
# Creating MicroHomology objects (from MicroHomology_module_v3)
microhomology_object = df.apply( lambda x: MicroHomology(x['Read'],
                                                            x['ALT'],
                                        x['REF'],
                                        x['reference_context_seq_120'],
                                        x['indel_pos'],
                                        x['indel_length'],
                                        x['indel_type']), axis = 1)


df2 = None
for i in microhomology_object.index:
    if (i == 0):
        df2 = microhomology_object[i].ex_data
    else:
        df2 = pd.concat([df2, microhomology_object[i].ex_data])
        
# print(df2)
df2.reset_index(inplace = True)
df = df.join(df2, how = 'right')
df.drop(columns = ['index'], inplace = True)

"""
for insertions: the motif is the repeat ('tamplate_switch_repeat')
for  mmej snap back: the motif is the repeat ('S_tamplate_switch_repeat')
for deletions: the motif is the 'mmej_cand'
for all indels the last nucleotide is the one that comes right before the DSB on the reference seq
"""

# creating empty columns for the probabilities
df['mmej_probability'] = np.nan
df['SD_mmej_snap_back_prob'] = np.nan
# reassining 'None' values to be False when needed for ferther filtrarion
df['mmej'].replace(to_replace = [None], value = False, inplace = True)
df['SD_mmej_trans'].replace(to_replace = [None], value = False, inplace = True)
df['SD_mmej_snap_back'].replace(to_replace = [None], value = False, inplace = True)

"""
The probability to find mmej will be calculate seperatly since each scenario has a different
motif. Other then the motif itself, the rest is similar.
"""
# creating a df with only deletions
df_with2K_del_mmej = df[((df['mmej'] == True) & 
           (df['indel_type'] == 'DEL'))].copy()

print('DEL mmej probability (Markov) starts', datetime.datetime.now())
df.loc[
    ((df['mmej'] == True) & (df['indel_type'] == 'DEL')),
    'mmej_probability'] = df_with2K_del_mmej.apply(lambda x: motif_probabily_calc(_sequance = x['reference_context_seq_2K'] , 
                                 _motif = x['mmej_cand'],
                                _indel_len = int(x['indel_length']),
                    _memory_dimer = x['last_dimer']) , axis = 1 )
print('DEL mmej probability (Markov) ends', datetime.datetime.now())

# creating a df with only insertions
df_with2K_ins_mmej = df[(df['SD_mmej_trans'] == True)].copy()
df_with2K_ins_mmej.loc[:,'mmej_cand_len'] = df_with2K_ins_mmej['mmej_cand_len'].astype('int')
print('INS mmej probability (Markov) starts', datetime.datetime.now())
df.loc[
    (df['SD_mmej_trans'] == True),
    'mmej_probability'] = df_with2K_ins_mmej.apply(lambda x: motif_probabily_calc(_sequance = x['reference_context_seq_2K'] , 
                                 _motif = x['tamplate_switch_repeat'],
                                _indel_len = x['mmej_cand_len'],
                    _memory_dimer = x['last_dimer']), axis = 1 )
print('INS mmej probability (Markov) ends', datetime.datetime.now())

# creating a df with only mmej snap back
df_with2K_mmej_snap_back = df[df['SD_mmej_snap_back'] == True].copy()
df_with2K_mmej_snap_back.loc[:,'S_mmej_cand_len'] = df_with2K_mmej_snap_back['S_mmej_cand_len'].astype('int')
print('snap back mmej probability (Markov) starts', datetime.datetime.now())
df.loc[
    (df['SD_mmej_snap_back'] == True),
    'SD_mmej_snap_back_prob'] = df_with2K_mmej_snap_back.apply(
    lambda x: motif_probabily_calc(_sequance = x['reference_context_seq_2K'] , 
                                 _motif = x['S_tamplate_switch_repeat'],
                                _indel_len = x['S_mmej_cand_len'],
                    _memory_dimer = x['S_last_dimer']), axis = 1 )
print('snap back mmej probability (Markov) ends', datetime.datetime.now())
# print(df.head())


"""
Sice the Markov model calculates the probability to find 2 motifs in a
defined range and a given context, this probability is the probability
that something isn't MMEJ, therefore if one wants to have the probability
of something to be a MMEJ, he should do 1-probability
"""
df['mmej_probability'] = 1 - df['mmej_probability']
df['SD_mmej_snap_back_prob'] = 1 - df['SD_mmej_snap_back_prob']

print(df.info())

# create a shuffeled data (to simulate real-life situation)
df = df.sample(df.shape[0], random_state = 0)
df

DEL mmej probability (Markov) starts 2022-02-28 11:05:58.107762
DEL mmej probability (Markov) ends 2022-02-28 11:05:59.853900
INS mmej probability (Markov) starts 2022-02-28 11:05:59.858314
INS mmej probability (Markov) ends 2022-02-28 11:06:04.274424
snap back mmej probability (Markov) starts 2022-02-28 11:06:04.275876
snap back mmej probability (Markov) ends 2022-02-28 11:06:07.098968
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype   
---  ------                     --------------  -----   
 0   REF                        160 non-null    object  
 1   ALT                        160 non-null    object  
 2   indel_type                 160 non-null    category
 3   indel_length               160 non-null    int64   
 4   indel_pos                  160 non-null    int64   
 5   Read                       160 non-null    object  
 6   reference_context_seq_2K   160 non-null    o

,REF,ALT,indel_type,indel_length,indel_pos,Read,reference_context_seq_2K,reference_context_seq_120,mmej,mmej_cand,...,SD_mmej_snap_back,S_tamplate_switch_repeat,S_true_rep_micro_rep,S_microhomology_marked,S_last_dimer,S_mmej_cand,mmej_cand_len,S_mmej_cand_len,mmej_probability,SD_mmej_snap_back_prob
110,ATGTACTTA,-,DEL,9,119,TTAAGAGGCTGTTGAGCCAAATAATCTGCTAAGGCGCTTCCTTTTATCGCCTTTTGGGTGACATAGACTATATCAAATTCGGATAGCAGGACTTGCCACCGGGCGATTCGTCCCGTGAGACCGGGTCCATCTTGGATATCAACCAGGTACTTAACCGGGTCCATCTTGGATATCAACCAGGTAGTATGGCTCAGCATGTACTACCTTAGGCGATGGGATGCCCATACTAAAGCACAACA,GTGGCATTTCCTCACATGGACACAATAATCATTTTCCATGGTAAGCCAGTAATAACCTGCTCTTAGGATCTTCCTGGCCATAGCATGCCCGTTGGCGTGTGTTCCAAACGAGCCCTCATGGACTTCCTCGATCATGTGATTTGCCTCCCTGGCATCCACACACCGCAGGAGTGTCATGTCGTGATTTCTCTTATACAGTATGCTTCCGCTCATGAAGAAACCGGCTGCCAACCTTCTCAATGTCCTCTTATCATTGTCGGCAATCTCTGGCGGATATTCTTTGCTTACGACATATCGCTTGATGTCGAAATACCAAGGCTTTCCGTCCCGTTCCTCTTCCACTTGGCAACAATGCGCGGGTTTGCCACGACACCAAAATTCAATGTAGGGTAGATCCCCGTGCGGTGTTAGCTGGAACATAGACGCCAAAGTAGCAAGTGCATCCGCCATTTGATTTTCCTCGCAGGGAACATGATGGAAGGAGATCTCATCGAAGGTCTTAGCCAATTCCTTGATATAGGCTTTGTAGGGTATCAGCTTGGGATCTCTAGTTTCCCATTCCCCTTTCAGCTGATGGATTACCAATGCCGAGTCGCCGTACACCTTGAGTAGTTTGACATTGGAGTCAATCGCTGCCTGGACGGCTAGGGCACATGCTTCATATTCGGCCATGTTGTTGGTGTAGTCGAATCCTAGCCTGGCTGTGAAGGGTACACATTGATTGTCCGGAGAGACCAATACTGCCCCAACGCCATGACCTAGAATGTTTGACGCTCCGTCAAACCATACGGTCCATTTGTCCCGATCTTCGTCCAATTTTTCCTCAAACAAGGCCATGATGTCCTCATCCGGGAATTCAGGATGCATGGGCTGATAGTCGTTAAGAGGCTGTTGAGCCAAATAATCTGCTAAGGCGCTTCCTTTTATCGCCTTTTGGGTGACATAGACTATATCAAATTCGGATAGCAGGACTTGCCACCGGGCGATTCGTCCCGTGAGATGTACTTAACCGGGTCCATCTTGGATATCAACCAGGTACTTAACCGGGTCCATCTTGGATATCAACCAGGTAGTATGGCTCAGCATGTACTACCTTAGGCGATGGGATGCCCATACTAAAGCACAACACGTTCTTTCGAGCAAGGAGTAATTCATCTCACAGGTCGTGAACTTCTTACTTAGGTAGTAAACAGCGCGCTCTTTCTTCTCGGATTCGTCATGTTGCCCCAGCATACACCCCATTGACTCGTCTAAGATTGTCATGTACAAGATGAGAGGCCTTCCAGGTACTGGTGGCATAAGCACGGGAGGATTCATAAGGCACTTCTTGATCCTTCCAAAAGCCTCTTGGCAATCCTCATTCCAGCGATCAGTTTGGTTTTTACGTAAGAGCTTGAACAACGGCTCACAAATGGCGGTGAGCTGCGATATGAATCTGGCAATATAATTCAAGCATCCCAGGAAACCTCGGACTTGCCTCTCTGTACGGGGTTCTGGCATCTCAAGGATAGCCTTCACCTTTTTGGGGTCTACCTCTATCCCTTTCTGGCTTACAATGAAACCAAGCAATTTCCCTGATTTGACCCCAAAGGTACACTTAGCGGGGTTCAACCTTAATTGATATTTCTTAAGCCTTTCGAACAACTTCCGCAGGTTGACAAGGTGTTCTTCCTCGGATTTAGATTTAGCAATTATGTCGTCCACGTAGACCTCGATCTCTTGATGCATCATATCATGGAACAAAGCTACCATGGCCCGTTGATAAGTTGCCCCGGCATTCTTGAGTCCAAAGGACATCACCTTGTAACAGAACGTCTCCCACAGGGTGACGAAAGTAGTCTTTTCCATATCCTCGGGCGCCATCTTTATCTGATTGTAACCGGAGAAACCGTCCATGAAGGAAAATAAAGCGAAATTGGCCGTGTTATCTACGAGGATATCGATGTGTGGTAAAGGAAAATTGTCCTTGGGACTGGCCCGATTCAGGTCCCGATAATCCACACACATT,TTAAGAGGCTGTTGAGCCAAATAATCTGCTAAGGCGCTTCCTTTTATCGCCTTTTGGGTGACATAGACTATATCAAATTCGGATAGCAGGACTTGCCACCGGGCGATTCGTCCCGTGAGATGTACTTAACCGGGTCCATCTTGGATATCAACCAGGTACTTAACCGGGTCCATCTTGGATATCAACCAGGTAGTATGGCTCAGCATGTACTACCTTAGGCGATGGGATGCCCATACTAAAGCACAACA,False,NaN,...,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
112,AATATATATATAT,-,DEL,13,119,TAATTTGTGATTTTTGATAAATTTCAATTAATAATAAAAAGTATGTTACAAGTATTTTTATAACATCGTAAATTGGTGGAGTTGTTTACAAATTCAACTTCAATTAAGTTTTTGTTTTTATATATATATATATATATATATATATATATATATATATATATATATATATATATATATATATATAATCTCTTTATGTCCTTGCTTTATATCATATCTTTTAACATATATAAGAATACGTA,GATAAGCAACAAAACGTGATCATTCAGTTTCCATGTTTAACTCGCATTTTAATTAAATAAGGTTTATAAAACAAAATTATATCGTATTTCTAAAAACGTTACCAAAAAATATCGTAATTCTAACCAAAACAGCTTTCCATTGGTTTTACATGTTTATGGAGTAATACTATACAGTACTATATTATCTCGAAAAACCAAAATTTGGTATTCAAATAGGGGACTGCCGCAATTTTTGACTGTGTGGTGATAGCATAGGAAAATTATTTTCATATTTGAAAATAAAAACATAAAGAAAACCACGTTTGGCATACCACTAGTATTTCACTTCAATTTTCAAAAGTTTTCATATTTGATATGATGATAAACACTCATTATTCAAGATATACGTAAGATATTCAAGAATATGGAGTGTTCTATAATGCTGAGCATTTCAAAAGTGAACATTGAATATGTTTGGAATATCTCATAAACCATTACGAATTAGCTAAACATAAAACATCAGTTTTTTTTTTTTTAATTTATAACGAGAACATAAAACATAGCGTTGGTTTGATGAAAAAAACATAACAATGGAGTTAACGTTTTAGGCTTGAGGTAACAAAAAAAATTAAGATTAAAAAATTAAGATCGATGTTTTTTCTTTACAGAAAATTAAGACTTAAAGTTATGACCTTTTATTGAAGCATTTATATACTATACGATTCAAAATATTGGTAAAAAAATTTCAGTGTGTGTTTGAAGCAGTGGATGAGGGATTGATTGATTTCAAGTACGTTAATATACTTTTTTTAATACATTTTATATTATTGACCAAAATGTATTAAAAATTGTAAAATCATCAGCAATACT

# Explenation about the output
## column name (column type) -> explenation
mmej (Boolian): Indicate weather an indel is a MMEJ or not (both for delerion and insertions (only trans mmej)). <br>
SD_mmej_trans (Boolian): Indicate weather an indel is a trans-MMEJ (insertion with two identical repeats flanking an MMEJ sequence).<br>
SD_mmej_snap_back (Boolian): Indicate weather an indel is a Snap-Back-MMEJ (insertion with one "regular" repeat and one reversed repeat flanking an MMEJ sequence).<br>
mmej_cand (str): The potential MMEJ sequence itself (This is an approximation).<br>
mmej_cand_len (int): Length of mmej_cand <br>
S_mmej_cand (str): The potential Snap-Back MMEJ sequence itself (This is an approximation).<br>
S_mmej_cand_len (int): length of S_mmej_cand <br>
tamplate_switch_repeat (str): The repeat portion of the Repeat -> mmej_cand -> Repeat pattern.<br>
S_tamplate_switch_repeat (str): The repeat portion of the Repeat -> mmej_cand -> Repeat pattern in Snap-Back MMEJ.<br>
true_rep_micro_rep_INS (str): The motif that we are looking for in a tamplate switch MMEJ (insertions only): Repeat -> mmej_cand -> Repeat. <br>
S_true_rep_micro_rep (str): The motif that we are looking for in a Snap-Back MMEJ (insertions only): Repeat -> mmej_cand -> Repeat. <br>
mmej_marked (str): The MMEJ marked on the accession/Read itself (marked as follow: upstream -> *[mmej_cand]* -> downstream). <br>
S_microhomology_marked (str): The Snap-Back MMEJ marked on the accession/Read itself (marked as follow: upstream -> *[mmej_cand]* -> downstream). <br>
mmej_marked_on_ref (str): The MMEJ marked on the reference genome (relevant only for deletions). <br>
last_dimer (str): This column is the dimer at the 0th state of the Markov model. <br>
S_last_dimer (str): This column is the dimer at the 0th state of the Markov model (for Snap-Back MMEJ). <br>
mmej_probability (float): The probability that a MMEJ is a real one and didn't occure by chance (a product of the Markov model) <br>
SD_mmej_snap_back_prob (float): The probability that a Snap-Back MMEJ is a real one and didn't occure by chance (a product of the Markov model) <br>

# Part B: Fake examples of the MMEJ detection process.
Fake and easy to grasp cases of different MMEJ types that the algorithm finds.

In [4]:
# Creating a fake dataset with fake examples
fake_df = pd.DataFrame(columns = ['ALT','REF', 'indel_type', 'indel_length', 'indel_pos', 
                                  'Read', 'reference_context_seq_2K', 'reference_context_seq_120'])

# Example 1: Deletion with high probability to be MMEJ
fake_read = 'C'*115 + 'AGGT' + 'C'*119
fake_ref_2K = 'C'*996 + 'AGGT' + 'CGGCC' + 'AGGT' + 'C'*991
fake_ref_120 = 'C'*115 + 'AGGT' + 'CGGCC' + 'AGGT' + 'C'*110
fake_df.loc[0, ['ALT','REF', 'indel_type', 'indel_length','indel_pos',
               'Read', 'reference_context_seq_2K', 'reference_context_seq_120']] =  '','CCCCCAGGT', 'DEL',len('CCCCCAGGT'), 119, fake_read, fake_ref_2K, fake_ref_120
# Example 2: Deletion with low probability to be MMEJ
fake_read = 'C'*20+'AGGT'*20+'C'*15 + 'AGGT' + 'C'*121

fake_ref_2K = 'C'*881 + 'C'*20+'AGGT'*20+'C'*15 + 'AGGT' + 'AGGT'*1 + 'C'*20 + 'AGGT'*10 + 'C'*936

fake_ref_120 = 'C'*20+'AGGT'*20+'C'*15 + 'AGGT' + 'AGGT'*1 + 'C'*20 + 'AGGT'*10 + 'C'*57

fake_df.loc[1, ['ALT','REF', 'indel_type', 'indel_length','indel_pos',
               'Read', 'reference_context_seq_2K', 'reference_context_seq_120']] =  '','AGGT', 'DEL',len('AGGT'), 119, fake_read, fake_ref_2K, fake_ref_120

# Example 2: Insertion (Trans) with low probability to be MMEJ
fake_read = 'C'*115 + 'AGGT' +'CCGGTCC' + 'AGGT' + 'C'*110
fake_ref_2K = 'C'*996 + 'CGGCCAGGT' + 'C'*991
fake_ref_120 = 'C'*119 + 'CGGCCAGGT' + 'C'*110

fake_df.loc[2, ['ALT','REF', 'indel_type', 'indel_length','indel_pos',
               'Read', 'reference_context_seq_2K', 'reference_context_seq_120']] =  'CCCCCAGGT','', 'INS',len('CCCCCAGGT'), 119, fake_read, fake_ref_2K, fake_ref_120

# Example 3: Insertion (Trans) with High probability to be MMEJ
fake_read = 'C'*115 + 'AGGT' +'CCAACGGGTCC' + 'AGGT' + 'C'*110
fake_ref_2K = 'C'*996 + 'CGGCCAGGT' + 'C'*991
fake_ref_120 = 'C'*119 + 'CGGCCAGGT' + 'C'*110

fake_df.loc[3, ['ALT','REF', 'indel_type', 'indel_length','indel_pos',
               'Read', 'reference_context_seq_2K', 'reference_context_seq_120']] =  'CCCCCAGGT','', 'INS',len('CCCCCAGGT'), 119, fake_read, fake_ref_2K, fake_ref_120

# Example 4: Insertion (Snap back + trans) with High probability to be MMEJ
fake_read = 'C'*119 +  'AGGT' + 'TGGA' +'C'*110
fake_ref_2K = 'C'*1000 + 'AGGT' + 'C' *995
fake_ref_120 = 'C'*119 + 'AGGT' + 'C' *110

fake_df.loc[4, ['ALT','REF', 'indel_type', 'indel_length','indel_pos',
               'Read', 'reference_context_seq_2K', 'reference_context_seq_120']] =  'TGGA','', 'INS',len('TGGA'), 119, fake_read, fake_ref_2K, fake_ref_120

# Example 5: Insertion (Snap back + trans) with Low probability to be MMEJ
fake_read = 'AGGAGCCC'*10+'C'*39 +  'AGGT' + 'TGGA' +'C'*20 + 'AGGAGCCC'*10 + 'C'*13
fake_ref_2K = ('AGGAGCCC'*10+'C'*20)*9 + 'C'*100 + 'AGGT' + ('AGGAGCCC'*10+'C'*20)*9 +'C' *96
fake_ref_120 = 'AGGAGCCC'*10+'C'*39 + 'AGGT' + 'C' *117

fake_df.loc[5, ['ALT','REF', 'indel_type', 'indel_length','indel_pos',
               'Read', 'reference_context_seq_2K', 'reference_context_seq_120']] =  'TGGA','', 'INS',len('TGGA'), 119, fake_read, fake_ref_2K, fake_ref_120
fake_df
df = fake_df

In [5]:
# %%timeit
"""
Detecting microhomology:
The data must have the following columns:
# accession_context -> A sequence with the indel and no gaps (type: str)
# derived_allele -> The indel itself (i.e. the sequence that got inserted/deleted)
# ancesstral_allele -> The reference sequence at the indel position 
    (in a case of insertion the value should be: "", in case of deletion 
    the value should be: the deleted sequence)
# reference_context_seq_120 -> the reference genome at a window of +-120bp from the indel position
    (i.e. -120bp ->INDEL->+ 120bp)
# indel_pos -> the indel position in the read
# indel_length -> the length of the indel
# indel_type -> the type of the indel (INS\DEL)
"""
microhomology_object = df.apply( lambda x: MicroHomology(x['Read'],
                                                            x['ALT'],
                                        x['REF'],
                                        x['reference_context_seq_120'],
                                        x['indel_pos'],
                                        x['indel_length'],
                                        x['indel_type']), axis = 1)


df2 = None
for i in microhomology_object.index:
    if (i == 0):
        df2 = microhomology_object[i].ex_data
    else:
        df2 = pd.concat([df2, microhomology_object[i].ex_data])
        
# print(df2)
df2.reset_index(inplace = True)
df = df.join(df2, how = 'right')
df.drop(columns = ['index'], inplace = True)

In [6]:
# %%timeit
"""
for insertions: the motif is the repeat ('tamplate_switch_repeat')
for  mmej snap back: the motif is the repeat ('S_tamplate_switch_repeat')
for deletions: the motif is the 'mmej_cand'
for all indels the last nucleotide is the one that comes right before the DSB on the reference seq
"""
# creating empty columns for the probabilities
df['mmej_probability'] = np.nan
df['SD_mmej_snap_back_prob'] = np.nan
# reassining 'None' values to be False when needed for ferther filtrarion
df['mmej'].replace(to_replace = [None], value = False, inplace = True)
df['SD_mmej_trans'].replace(to_replace = [None], value = False, inplace = True)
df['SD_mmej_snap_back'].replace(to_replace = [None], value = False, inplace = True)
"""
The probability to find mmej will be calculate seperatly since each scenario has a different
motif. Other then the motif itself, the rest is similar.
"""
# creating a df with only deletions
df_with2K_del_mmej = df[((df['mmej'] == True) & 
           (df['indel_type'] == 'DEL'))].copy()

print('DEL mmej probability (Markov) starts', datetime.datetime.now())
df.loc[
    ((df['mmej'] == True) & (df['indel_type'] == 'DEL')),
    'mmej_probability'] = df_with2K_del_mmej.apply(lambda x: motif_probabily_calc(_sequance = x['reference_context_seq_2K'] , 
                                 _motif = x['mmej_cand'],
                                _indel_len = int(x['indel_length']),
                    _memory_dimer = x['last_dimer']) , axis = 1 )
print('DEL mmej probability (Markov) ends', datetime.datetime.now())

# creating a df with only insertions
df_with2K_ins_mmej = df[(df['SD_mmej_trans'] == True)].copy()
df_with2K_ins_mmej.loc[:,'mmej_cand_len'] = df_with2K_ins_mmej['mmej_cand_len'].astype('int')
print('INS mmej probability (Markov) starts', datetime.datetime.now())
df.loc[
    (df['SD_mmej_trans'] == True),
    'mmej_probability'] = df_with2K_ins_mmej.apply(lambda x: motif_probabily_calc(_sequance = x['reference_context_seq_2K'] , 
                                 _motif = x['tamplate_switch_repeat'],
                                _indel_len = x['mmej_cand_len'],
                    _memory_dimer = x['last_dimer']), axis = 1 )
print('INS mmej probability (Markov) ends', datetime.datetime.now())

# creating a df with only mmej snap back
df_with2K_mmej_snap_back = df[df['SD_mmej_snap_back'] == True].copy()
df_with2K_mmej_snap_back.loc[:,'S_mmej_cand_len'] = df_with2K_mmej_snap_back['S_mmej_cand_len'].astype('int')
print('snap back mmej probability (Markov) starts', datetime.datetime.now())
df.loc[
    (df['SD_mmej_snap_back'] == True),
    'SD_mmej_snap_back_prob'] = df_with2K_mmej_snap_back.apply(
    lambda x: motif_probabily_calc(_sequance = x['reference_context_seq_2K'] , 
                                 _motif = x['S_tamplate_switch_repeat'],
                                _indel_len = x['S_mmej_cand_len'],
                    _memory_dimer = x['S_last_dimer']), axis = 1 )

print('snap back mmej probability (Markov) ends', datetime.datetime.now())
"""
Sice the Markov model calculates the probability to find 2 motifs in a
defined range and a given context, this probability is the probability
that something isn't MMEJ, therefore if one wants to have the probability
of something to be a MMEJ, he should do 1-probability
"""
df['mmej_probability'] = 1 - df['mmej_probability']
df['SD_mmej_snap_back_prob'] = 1 - df['SD_mmej_snap_back_prob']
pd.set_option("display.max_columns", None)
df

DEL mmej probability (Markov) starts 2022-02-28 11:06:07.237624
DEL mmej probability (Markov) ends 2022-02-28 11:06:07.319061
INS mmej probability (Markov) starts 2022-02-28 11:06:07.320087
INS mmej probability (Markov) ends 2022-02-28 11:06:07.577288
snap back mmej probability (Markov) starts 2022-02-28 11:06:07.578760
snap back mmej probability (Markov) ends 2022-02-28 11:06:07.596421


,ALT,REF,indel_type,indel_length,indel_pos,Read,reference_context_seq_2K,reference_context_seq_120,mmej,mmej_cand,true_rep_micro_rep_INS,mmej_marked,mmej_marked_on_ref,tamplate_switch_repeat,last_dimer,SD_mmej_trans,SD_mmej_snap_back,S_tamplate_switch_repeat,S_true_rep_micro_rep,S_microhomology_marked,S_last_dimer,S_mmej_cand,mmej_cand_len,S_mmej_cand_len,mmej_probability,SD_mmej_snap_back_prob
0,,CCCCCAGGT,DEL,9,119,CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCAGGTCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC,CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCAGGTCGGCCAGGTCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC,CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCAGGTCGGCCAGGTCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC,True,CCAGGT,NaN,CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC*[CCAGGT]*CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC,CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC*[CCAGGT]*|CGG*[CCAGGT]*CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC,NaN,GT,False,False,NaN,NaN,NaN,NaN,NaN,6,NaN,9.995146e-01,NaN
1,,AGGT,DEL,4,119,CCCCCCCCCCCCCCCCCCCCAGGTAGGTAGGTAGGTAGGTAGGTAGGTAGGTAGGTAGGTAGGTAGGTAGGTAGGTAGGTAGGTAGGTAGGTAGGTAGGTCCCCCCCCCCCCCCCAGGTCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC,CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC

# Part C: Proving the Markov chain integrety.
<font size="3"> Showing different cases of calculating the probability that something is a real MMEJ in a given context. <br>
Each example has an expected result, and then shows that the MMEJ_2nd_order_MM_v2 module calculate <br>
the expected result. <br>


<font size="5"> A pre calculated example 1: <br> 
<font size="3"> to make sure that the Markov chain works as expected:
Bellow, the folloeing sequence: ACTTACGG<br>
in this sequence, given state0 is all zeros except AC (memory_dimer = 'AC')
AC is followed by either TT or GG.<br>
therefore, the probability to find TT, given AC, should be equal to the probability to find GG.

In [7]:
# Creating a fake sequence
fasta_context_seq_2K = 'ACTTACGG'*100000
# setting parameters
memory_dimer = 'AC'
indel_length = 1
mmej_cand = 'T'
# calling motif_probabily_calc for T
TT = motif_probabily_calc(_sequance = fasta_context_seq_2K , 
                                 _motif = mmej_cand,
                                _indel_len = indel_length,
                    _memory_dimer = memory_dimer) # 0.500000499995 

mmej_cand = 'G'
# calling motif_probabily_calc for G
GG = motif_probabily_calc(_sequance = fasta_context_seq_2K , 
                                 _motif = mmej_cand,
                                _indel_len = indel_length,
                    _memory_dimer = memory_dimer) # 0.500000499995 

print(f'P(T|AC): {TT} \nP(G|AC): {GG}')

P(T|AC): 0.500000499995 
P(G|AC): 0.500000499995


<font size="5"> A pre calculated example 2: <br>
<font size="3"> to make sure that the Markov chain works as expected:
Bellow, the folloeing sequence: AATTAACCAAGG <br>
in this sequence, given state0 is all zeros except AA (memory_dimer = 'AA')
AA is followed by either TT, CC or TT. <br>
therefore, the probability to find TT given AA, should be equal to the probability to find CC or GG.

In [8]:
fasta_context_seq_2K = 'AATTAACCAAGG'*100000
memory_dimer = 'AA'
indel_length = 1

mmej_cand = 'C'
C = motif_probabily_calc(_sequance = fasta_context_seq_2K , 
                                 _motif = mmej_cand,
                                _indel_len = indel_length,
                    _memory_dimer = memory_dimer) # 0.33333366666555553 

mmej_cand = 'G'
G = motif_probabily_calc(_sequance = fasta_context_seq_2K , 
                                 _motif = mmej_cand,
                                _indel_len = indel_length,
                    _memory_dimer = memory_dimer) # 0.33333366666555553 

mmej_cand = 'T'
T = motif_probabily_calc(_sequance = fasta_context_seq_2K , 
                                 _motif = mmej_cand,
                                _indel_len = indel_length,
                    _memory_dimer = memory_dimer) # 0.33333366666555553 

print(f'P(C|AA): {C} \nP(G|AA): {G} \nP(T|AA): {T}')

P(C|AA): 0.33333366666555553 
P(G|AA): 0.33333366666555553 
P(T|AA): 0.33333366666555553


<font size="5"> 
A pre calculated example 3: <br>
<font size="3"> Bellow, the folloeing sequence: TTCCGG <br>
in this sequence, given state0 is all zeros except TT (memory_dimer = 'TT')
TT is followed only by CC.
therefore, the probability to find CC should be equal to 1, while the probability to find GG or TT <br>
should be equal to 0 (or < 1**-6).


In [9]:
fasta_context_seq_2K = 'TTCCGG'*100000

memory_dimer = 'TT'
indel_length = 1

mmej_cand = 'C'
C = motif_probabily_calc(_sequance = fasta_context_seq_2K , 
                                 _motif = mmej_cand,
                                _indel_len = indel_length,
                    _memory_dimer = memory_dimer) # 1.0000009999700001 

mmej_cand = 'G'
G = motif_probabily_calc(_sequance = fasta_context_seq_2K , 
                                 _motif = mmej_cand,
                                _indel_len = indel_length,
                    _memory_dimer = memory_dimer) # 1.000000082740371e-11

mmej_cand = 'T'
T = motif_probabily_calc(_sequance = fasta_context_seq_2K , 
                                 _motif = mmej_cand,
                                _indel_len = indel_length,
                    _memory_dimer = memory_dimer) # 1.000000082740371e-11

print(f'P(C|TT): {C} \nP(G|TT): {G} \nP(T|TT): {T}')

P(C|TT): 1.0000009999700001 
P(G|TT): 1.000000082740371e-11 
P(T|TT): 1.000000082740371e-11


<font size="5"> A pre calculated example 4: <br>
<font size="3"> Bellow, the folloeing sequence: TTTT <br>
in this sequence, given state0 is all zeros except TT (memory_dimer = 'TT')
TT is followed only by T. <br>
therefore, the probability to find CC,GG,AA should be all equal to zero, 
the probability to find TT shold be equal to 1.

In [10]:
fasta_context_seq_2K = 'TTTT'*100000

memory_dimer = 'TT'
indel_length = 1

mmej_cand = 'C'
CC = motif_probabily_calc(_sequance = fasta_context_seq_2K , 
                                 _motif = mmej_cand,
                                _indel_len = indel_length,
                    _memory_dimer = memory_dimer) # 2.5000002068509275e-12 

mmej_cand = 'G'
GG = motif_probabily_calc(_sequance = fasta_context_seq_2K , 
                                 _motif = mmej_cand,
                                _indel_len = indel_length,
                    _memory_dimer = memory_dimer) # 2.5000002068509275e-12 

mmej_cand = 'A'
AA = motif_probabily_calc(_sequance = fasta_context_seq_2K , 
                                 _motif = mmej_cand,
                                _indel_len = indel_length,
                    _memory_dimer = memory_dimer) # 2.5000002068509275e-12 


mmej_cand = 'T'
TT = motif_probabily_calc(_sequance = fasta_context_seq_2K , 
                                 _motif = mmej_cand,
                                _indel_len = indel_length,
                    _memory_dimer = memory_dimer) # 1.0000009999924997



print(f'P(C|TT): {CC} \nP(A|TT): {AA} \nP(G|TT): {GG} \nP(T|TT): {TT}')

P(C|TT): 2.5000002068509275e-12 
P(A|TT): 2.5000002068509275e-12 
P(G|TT): 2.5000002068509275e-12 
P(T|TT): 1.0000009999924997
